# Aula 8 — N-grams e engenharia de features textuais

**Quando uma sequência de palavras informa mais do que palavras isoladas**

Nas aulas anteriores, representamos documentos com Bag-of-Words e TF-IDF. Essas técnicas tratam cada termo como uma feature independente.

Mas linguagem também depende de combinações.

Compare:

```text
gostei do atendimento
não gostei do atendimento
```

A palavra `gostei` aparece nas duas frases, mas o par `não gostei` muda completamente a interpretação.

Nesta aula vamos estudar **n-grams** e usá-los como uma forma simples e poderosa de engenharia de features para texto.


## 1. Objetivos de aprendizagem

Ao final desta aula, você deverá ser capaz de:

- explicar o que são unigramas, bigramas e trigramas;
- gerar n-grams com `CountVectorizer` e `TfidfVectorizer`;
- interpretar `ngram_range`;
- comparar representações com palavras isoladas e sequências;
- entender o ganho e o custo de aumentar o espaço de features;
- reconhecer n-grams como uma forma de Feature Engineering aplicada a texto.


## 📘 Glossário da aula

Conceitos centrais: **n-gram · unigram · bigram · trigram · ngram_range · espaço de features · dimensionalidade**.

- [Glossário PT-BR](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md)
- [Glossary EN](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.en.md)

> Use o glossário para consolidar o vocabulário técnico; o notebook continua autocontido para aprendizagem.


## Antes de começar — trabalhe na sua própria cópia

Crie sua cópia do notebook no Kaggle antes de executar ou modificar qualquer célula.


## 2. O que é um n-gram?

Um **n-gram** é uma sequência de `n` tokens consecutivos.

Para a frase:

```text
não gostei do atendimento
```

temos, por exemplo:

- unigramas: `não`, `gostei`, `do`, `atendimento`;
- bigramas: `não gostei`, `gostei do`, `do atendimento`;
- trigramas: `não gostei do`, `gostei do atendimento`.

Aumentar `n` preserva mais contexto local, mas também aumenta o número de possíveis features.


## 3. Começando com unigramas

Primeiro vamos observar uma representação tradicional com palavras isoladas.


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd

texts = [
    "gostei do atendimento",
    "não gostei do atendimento",
    "não gostei da demora",
]

unigram_vectorizer = CountVectorizer(ngram_range=(1, 1))
X_unigram = unigram_vectorizer.fit_transform(texts)

unigram_features = unigram_vectorizer.get_feature_names_out()

pd.DataFrame(
    X_unigram.toarray(),
    columns=unigram_features,
    index=["doc_1", "doc_2", "doc_3"],
)


### O que observar

A palavra `gostei` aparece tanto em uma mensagem positiva quanto em mensagens negativas.

O unigram `não` existe como outra feature, mas a relação local `não gostei` não aparece explicitamente.

**Checkpoint 1:** pense no que um modelo precisaria aprender para combinar essas duas features corretamente.


## 4. Adicionando bigramas

Agora vamos permitir unigramas **e** bigramas usando `ngram_range=(1, 2)`.


In [ ]:
bigram_vectorizer = CountVectorizer(ngram_range=(1, 2))
X_bigram = bigram_vectorizer.fit_transform(texts)

bigram_features = bigram_vectorizer.get_feature_names_out()

print("Número de features com unigramas:", len(unigram_features))
print("Número de features com unigramas + bigramas:", len(bigram_features))
print()
print(bigram_features)


### O que mudou?

Agora surgem features como:

- `não gostei`;
- `gostei do`;
- `do atendimento`.

Essas features preservam parte da estrutura local da frase.

Mas observe o preço: o número de colunas aumenta.


## 5. O trade-off: contexto versus dimensionalidade

N-grams maiores podem capturar padrões úteis, mas também:

- aumentam o espaço de features;
- tornam a matriz ainda mais esparsa;
- exigem mais dados para que sequências se repitam;
- podem capturar combinações acidentais muito específicas.

Esse é um exemplo clássico de trade-off de engenharia:

```text
mais contexto
↔
mais dimensionalidade e esparsidade
```


## 6. N-grams com TF-IDF

N-grams não pertencem apenas ao Bag-of-Words. Também podemos ponderá-los com TF-IDF.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_ngrams = TfidfVectorizer(ngram_range=(1, 2))
X_tfidf_ngrams = tfidf_ngrams.fit_transform(texts)

tfidf_features = tfidf_ngrams.get_feature_names_out()

tfidf_df = pd.DataFrame(
    X_tfidf_ngrams.toarray(),
    columns=tfidf_features,
    index=["doc_1", "doc_2", "doc_3"],
)

tfidf_df.round(3)


### O que interpretar

Agora temos duas decisões combinadas:

1. **quais sequências viram features** (`ngram_range`);
2. **como essas features recebem peso** (TF-IDF).

Isso mostra como uma representação textual é construída por várias escolhas de engenharia.


## 7. Comparando unigramas e bigramas em um classificador

Vamos usar validação cruzada para comparar duas configurações simples no mesmo pipeline.


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import cross_val_score

train_texts = [
    "gostei muito do atendimento",
    "atendimento muito bom",
    "serviço rápido e excelente",
    "não gostei do atendimento",
    "não gostei do serviço",
    "atendimento muito ruim",
    "não recomendo o serviço",
    "recomendo muito o atendimento",
]

train_labels = [
    "positivo", "positivo", "positivo", "negativo",
    "negativo", "negativo", "negativo", "positivo",
]

def evaluate_ngram_range(ngram_range):
    model = Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=ngram_range)),
        ("classifier", MultinomialNB()),
    ])
    scores = cross_val_score(model, train_texts, train_labels, cv=4, scoring="f1_macro")
    return scores

scores_uni = evaluate_ngram_range((1, 1))
scores_uni_bi = evaluate_ngram_range((1, 2))

print("Unigramas       :", scores_uni, "média =", round(scores_uni.mean(), 3))
print("Uni + bigramas  :", scores_uni_bi, "média =", round(scores_uni_bi.mean(), 3))


### Cuidado com a conclusão

Nosso dataset é minúsculo. O objetivo é aprender **como comparar representações**, não declarar uma vencedora universal.

Bigramas não são automaticamente melhores. Eles precisam ser avaliados no contexto do problema, do corpus e da quantidade de dados disponível.


## 8. Conexão com Feature Engineering

Na Aula 3, transformamos palavras em features. Na Aula 4, alteramos seus pesos com TF-IDF.

Agora estamos criando **novas features compostas** a partir de sequências de tokens.

Essa é uma conexão direta com o conceito geral de Feature Engineering:

```text
observação original
→ transformação
→ feature potencialmente mais informativa
```

Referência complementar: [Kaggle Learn — Feature Engineering](https://www.kaggle.com/learn/feature-engineering).


## 9. Exercício guiado

Use as mensagens abaixo:

```python
messages = [
    'não consigo acessar minha conta',
    'consigo acessar minha conta normalmente',
    'não consigo pagar a fatura',
]
```

Seu código deve:

1. criar um `TfidfVectorizer` com unigramas e bigramas;
2. transformar as mensagens;
3. exibir os nomes das features;
4. construir um `DataFrame` com a matriz TF-IDF;
5. verificar se `não consigo` aparece como feature.


In [ ]:
# Escreva sua solução aqui.

messages = [
    'não consigo acessar minha conta',
    'consigo acessar minha conta normalmente',
    'não consigo pagar a fatura',
]

# Continue a partir daqui.


### Dica e solução

Execute **uma vez** a próxima célula para preparar `q8.hint()` e `q8.solution()`.


In [ ]:
from IPython.display import Markdown, display

class TILExercise:
    def __init__(self, hint_text, solution_text):
        self._hint_text = hint_text
        self._solution_text = solution_text

    def hint(self):
        display(Markdown(f"### Dica\n\n{self._hint_text}"))

    def solution(self):
        display(Markdown(f"### Solução\n\n{self._solution_text}"))

q8 = TILExercise(
    hint_text=(
        "Use **scikit-learn + pandas**. A classe principal é `TfidfVectorizer`. "
        "Configure `ngram_range=(1, 2)`, use `fit_transform()`, `get_feature_names_out()` e depois `pd.DataFrame(...)`. "
        "Para testar a existência do bigrama, você pode usar `'não consigo' in features`."
    ),
    solution_text=(
        "Uma possível solução executável é:\n\n"
        "```python\n"
        "from sklearn.feature_extraction.text import TfidfVectorizer\n"
        "import pandas as pd\n\n"
        "vectorizer = TfidfVectorizer(ngram_range=(1, 2))\n"
        "X = vectorizer.fit_transform(messages)\n"
        "features = vectorizer.get_feature_names_out()\n\n"
        "display(pd.DataFrame(X.toarray(), columns=features).round(3))\n"
        "print('não consigo' in features)\n"
        "```"
    ),
)

print("Exercício preparado. Tente resolver antes de usar q8.hint() ou q8.solution().")


In [ ]:
# Remova o # da linha abaixo se quiser uma dica.
# q8.hint()


In [ ]:
# Remova o # da linha abaixo para revelar a solução.
# q8.solution()


## 10. Reprodutibilidade

- linguagem: Python;
- bibliotecas: `scikit-learn` e `pandas`;
- representações: CountVectorizer e TfidfVectorizer;
- n-grams: unigramas e bigramas;
- avaliação demonstrativa: validação cruzada com F1 macro;
- acelerador: CPU;
- internet: desabilitada;
- dataset externo: nenhum.


## 11. Resumo

Nesta aula, você aprendeu que:

- n-grams representam sequências consecutivas de tokens;
- unigramas usam palavras isoladas;
- bigramas e trigramas preservam mais contexto local;
- `ngram_range` controla quais tamanhos de sequência entram na representação;
- mais contexto aumenta também dimensionalidade e esparsidade;
- n-grams podem ser combinados com TF-IDF;
- criar n-grams é uma forma de engenharia de features textuais;
- representações devem ser comparadas experimentalmente, não escolhidas por intuição apenas.

### Ideia principal

```text
Às vezes, o significado útil não está em uma palavra,
mas na combinação de palavras que aparece ao redor dela.
```

**Fim da Aula 8.**
